<div align="center">

<a href="https://colab.research.google.com/github/DubeyDivyansh/video_production_engine/blob/main/quick_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Modular AI Video & Voice Production Studio
### Free Tesla T4 GPU (16GB VRAM) • Persistent Google Drive • Zero Cloud Cost

</div>

---

### 🌟 Capabilities:
1. **Continuous Avatar Knowledge Base:** Store your voice, portrait, and base video in Google Drive. As you upload new media, the system automatically evaluates SNR, sharpness, and motion stability to continuously upgrade your avatar profile without restarting from scratch.
2. **Zero-Shot Voice Cloning (F5-TTS):** Generates studio-grade cloned speech from as little as 5 seconds of clean reference voice.
3. **Dual Animation Engines:**
   - **LatentSync (Default):** Retains 100% of body language, natural gestures, and clothing for whole-body videos.
   - **LivePortrait / SadTalker:** High-fidelity facial performance animation from single portrait photos.
4. **Studio Post-Production Enhancers:**
   - **Background Matting & Replacement (BiRefNet / Chroma Key / Solid Studio Backdrops)**
   - **Face Super-Resolution & Restoration (GFPGAN v1.4)**
   - **Audio Spectral Denoising & -16 LUFS Mastering (EBU R128)**
   - **Dynamic BGM Mixing with Voice Ducking & Styled Subtitle Burn-In**
5. **Visual Node UI (ComfyUI Geeky LatentSync):** Optional 1-click Cloudflare tunnel for node-graph experimentation.

---
## 0. Hardware & GPU Environment Check
Verify your runtime has an active NVIDIA Tesla T4 GPU with ~15GB of free VRAM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

---
## 1. Mount Google Drive (Persistent Storage)
Mount Google Drive to persist your heavy model weights (~6 GB), avatar profiles, and generated deliverables across ephemeral Colab sessions.

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/video_gen_platform'
    for sub in ['checkpoints', 'avatars', 'inputs', 'outputs']:
        os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)
    print(f'✅ Persistent Drive storage active at: {BASE_DIR}')
except Exception as e:
    print('⚠️ Drive mount notice (running in local / ephemeral mode):', e)

---
## 2. Clone Repository & Install Dependencies
Clones `video_production_engine` from GitHub and installs the unified production dependencies.

In [ ]:
# 1. Clone repository from GitHub
!git clone https://github.com/DubeyDivyansh/video_production_engine.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app

# 2. Install production dependencies
!pip install -q -r requirements.txt

# 3. Ensure system ffmpeg is up to date
!apt-get install -y -q ffmpeg > /dev/null 2>&1

import sys
if '/content/app' not in sys.path:
    sys.path.append('/content/app')

print('✅ Production environment ready!')

---
## 3. Persistent Avatar Knowledge Base (Continuous Improvement)
Register your personal photos, videos, and audio samples under your avatar profile (`divyansh`).
The engine **automatically scores quality** (Audio SNR, image sharpness, video motion stability), archives raw assets, and updates optimal generation pointers on Google Drive.

In [ ]:
from src.avatar.manager import AvatarManager
import json

# Initialize Avatar Manager connected to Google Drive
manager = AvatarManager(use_drive=True)

# 1. Register portrait image
print('[*] Registering portrait image...')
!wget -q -O /content/my_face.jpeg https://files.catbox.moe/puyuek.jpeg
res_img = manager.register_media('divyansh', '/content/my_face.jpeg', media_type='image')
print(f"    - Sharpness Score: {res_img['quality_score']:.2f}")

# 2. Ingest and extract speech audio from Google Drive lecture video
DRIVE_LECTURE_URL = 'https://drive.google.com/file/d/1yCoOYz-Q-pzAaDYEWJUV1RihuiYJGQX2/view?usp=sharing'
print('\n[*] Ingesting user lecture video & extracting pristine speech audio from Google Drive...')
res_drive = manager.ingest_drive_url(
    avatar_id='divyansh',
    drive_url_or_id=DRIVE_LECTURE_URL,
    media_type='video',
    extract_video_audio=True
)

# 3. Inspect upgraded Avatar Profile state
profile = manager.get_avatar('divyansh')
print(f"\n🎉 Avatar Profile Active: '{profile.avatar_id}'")
print(f"   Optimal Audio: {profile.optimal_audio_path} (Score: {profile.optimal_audio_score:.2f})")
print(f"   Optimal Image: {profile.optimal_image_path} (Score: {profile.optimal_image_score:.2f})")
print(f"   Total Archived Assets: {len(profile.raw_assets)}")

---
## 4. Studio Video Production Run (Zero-Restart End-to-End Pipeline)
Run the complete studio production pipeline with 1 call:
1. Loads avatar assets from Drive.
2. Synthesizes cloned voice with F5-TTS.
3. Masters audio (spectral noise removal + -16 LUFS).
4. Enforces VRAM fence (`torch.cuda.empty_cache()` + `gc.collect()`).
5. Drives whole-body lip-sync (LatentSync with T4 FP16 patch) or facial reenactment.
6. Restores and upscales face via GFPGAN.
7. Composites custom background (image, video, or solid studio backdrop).
8. Burns styled subtitles & mixes background music with automatic speech ducking.
9. Saves finished deliverable to Google Drive!

In [ ]:
from src.pipeline import VideoProductionPipeline

# Initialize unified pipeline
pipeline = VideoProductionPipeline(use_drive=True)

# Script text for your avatar to deliver
SCRIPT = (
    "Hello! Welcome to our autonomous video production studio. "
    "Everything you are seeing and hearing is generated zero-cost on Google Colab, "
    "with continuous avatar calibration, studio audio mastering, and background replacement."
)

# Execute Full Production
final_video = pipeline.produce_video(
    avatar_id='divyansh',                   # Pulls optimal assets from Drive knowledge base
    script_text=SCRIPT,
    output_name='divyansh_studio_master.mp4',
    engine='sadtalker',                    # 'sadtalker' for photo, 'latentsync' for body video
    denoise_audio=True,                    # Studio spectral gating + EBU R128 loudness
    enhance_face=True,                     # GFPGAN facial super-resolution
    color_backdrop='#1A1A2E',              # Modern dark slate studio backdrop
    add_subtitles=True,                    # Auto-generate & burn styled subtitles
)

print(f"\n🎬 Production Deliverable Ready at: {final_video}")

---
## 5. Inline Video Player & Deliverable Verification
Preview the generated MP4 deliverable directly inside your Colab notebook.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

video_to_play = final_video if 'final_video' in locals() and os.path.exists(final_video) else None
if not video_to_play:
    fallback = '/content/drive/MyDrive/video_gen_platform/outputs/divyansh_studio_master.mp4'
    if os.path.exists(fallback):
        video_to_play = fallback

if video_to_play and os.path.exists(video_to_play):
    mp4_bytes = open(video_to_play, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <div style="text-align: center;">
      <h3>🎉 Rendered Deliverable: {os.path.basename(video_to_play)}</h3>
      <video width=640 controls autoplay loop style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
        <source src="{data_url}" type="video/mp4">
      </video>
    </div>
    '''))
else:
    print('⚠️ No video deliverable found to display. Please run Section 4 first.')

---
## 6. (Optional) Visual UI: ComfyUI + Geeky LatentSync Wrapper via Cloudflare Tunnel
For users who want to experiment with interactive node graphs in browser.
Launches ComfyUI and exposes a public HTTPS tunnel.

In [ ]:
# 1. Clone ComfyUI and Geeky LatentSync Wrapper if needed
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/GeekyGhost/ComfyUI-Geeky-LatentSyncWrapper.git 2>/dev/null || true

# 2. Install Cloudflare tunnel
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

# 3. Launch ComfyUI with low-VRAM flag in background and start tunnel
%cd /content/ComfyUI
import subprocess
import time

print('[*] Starting ComfyUI server...')
comfy_proc = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--lowvram'])
time.sleep(3)

print('[*] Exposing public Cloudflare tunnel...')
!cloudflared tunnel --url http://127.0.0.1:8188 > /content/tunnel.log 2>&1 &
time.sleep(5)

!grep -o 'https://.*\.trycloudflare\.com' /content/tunnel.log | head -n 1
print('\n👉 Click the .trycloudflare.com link above to open ComfyUI!')